# OTTO · Retrieval quality meets search cost

**Experiment 06 / Fold 0 ANN benchmark**

The saved two-tower model adds complementary candidates. This experiment asks whether approximate search retains that value at a useful search cost. The model, catalogue embeddings, and exact top-800 predictions are frozen; model training is not repeated.

**Evidence policy:** ANN measurements are pending until a verified managed report is available. The status below follows the actual report. Rerunning the notebook renders its quality, fidelity, latency, and resource evidence. No ANN performance values are simulated.

Use a separate analysis kernel with the pinned dependencies in `notebooks/requirements.txt`; the CPU control-plane lock and managed GPU profile remain separate.

[Operating guide](../docs/ANN_BENCHMARK.md) · [Configuration](../configs/two_tower_ann.toml) · [Previous measured results](05_two_tower_results.ipynb)

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import time
import tomllib

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import HTML, display

STARTED = time.perf_counter()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/two_tower_ann.toml").is_file())
configuration = tomllib.loads((ROOT / "configs/two_tower_ann.toml").read_text())["benchmark"]
OBJECTIVES = ("clicks", "carts", "orders")
COLORS = {"clicks": "#3366b0", "carts": "#17806e", "orders": "#c05b43"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "#f7f9fc", "axes.facecolor": "#f7f9fc"})

published = ROOT / "reports/metrics/two_tower_fold0_ann.json"
pointer_path = ROOT / "artifacts/two_tower_ann/latest.json"
report_path = published
pointer = None
# Prefer a downloaded tracked run while working locally; GitHub uses the committed report.
if pointer_path.is_file():
    pointer = json.loads(pointer_path.read_text())
    if pointer.get("report_path"):
        report_path = ROOT / pointer["report_path"]
report = json.loads(report_path.read_text()) if report_path.is_file() else None
if report is not None:
    assert report["status"] == "passed", "Report is not a completed experiment"
    if pointer is not None and report_path != published:
        receipt = json.loads(Path(str(report_path) + ".json").read_text())
        assert receipt["input_id"] == pointer["run_id"] == report["input_id"]
        assert hashlib.sha256(report_path.read_bytes()).hexdigest() == receipt["sha256"]
    print(f"report={report_path.relative_to(ROOT)}")
    print(f"run_id={report['input_id']}")
print(f"[{datetime.now(timezone.utc).isoformat()}] notebook_start "
      f"ann_measurements={'available' if report is not None else 'pending'}")


[2026-09-06T22:15:29.859380+00:00] notebook_start ann_measurements=pending


## Recorded execution diagnostics

Three managed attempts stopped during argument parsing, catalogue validation, and S3 stream restoration. The history retains each outcome and its billable time. The catalogue lookup now preserves trained embedding rows; artifact reads retain the SDK streaming wrapper and validate downloads before committing them.

Recovery checks use actual boto3 HTTP responses, including a complete small-model run restored into a fresh directory. A saved production S3 part was restored without recomputation. These checks establish recovery behavior at the tested scale; full-catalogue ANN measurements remain pending. The separate AWS image loader warning is still unresolved.


In [2]:
launch_path = ROOT / "reports/metrics/two_tower_fold0_ann_launch.json"
if launch_path.is_file():
    launch = json.loads(launch_path.read_text())
    attempts = [*launch.get("previous_attempts", []), launch]
    display(pd.DataFrame([{
        "Attempt": i + 1, "Status": row["status"], "Stopped stage": row["phase"],
        "AWS billable seconds": row["billable_seconds"],
        "Committed reference parts": row.get("committed_reference_parts", 0),
    } for i, row in enumerate(attempts)]).style.hide(axis="index"))
    print(launch["cause"])
    print(launch["interpretation"])
catalogue = json.loads((ROOT / "reports/metrics/two_tower_fold0_catalogue.json").read_text())
display(pd.DataFrame([
    ("Actual catalogue items validated", f"{catalogue["catalogue_items"]:,}"),
    ("IDs sorted", str(catalogue["ids_sorted"])),
    ("Embedding row order", catalogue["row_order"]),
    ("Catalogue check", catalogue["status"]),
    ("Reference count parts verified", catalogue["production_reference_reuse_check"]["parts_validated"]),
], columns=["Input contract", "Observed value"]).style.hide(axis="index"))

recovery = launch.get("recovery_validation", {})
if recovery:
    production = recovery["production_s3_restore"]
    display(pd.DataFrame([
        ("Root tests passed", recovery["root_tests_passed"]),
        ("Worker tests passed", recovery["worker_tests_passed"]),
        ("Worker Python warning policy", recovery["worker_python_warnings"]),
        ("Production S3 restore", production["status"]),
        ("Recomputed saved reference part", production["recomputed"]),
        ("AWS container loader warning", launch["container_warning"]["status"]),
    ], columns=["Recovery check", "Observed value"]).style.hide(axis="index"))


StreamingBody.__enter__ returned its raw HTTPResponse, which has no iter_chunks method. The previous test double returned itself and concealed this SDK behavior.
Catalogue preflight passed and all 96 compatible reference parts were imported. Restoration stopped before ANN indexing or evaluation. No full-scale ANN quality or latency was measured.


Attempt,Status,Stopped stage,AWS billable seconds,Committed reference parts
1,failed,managed_argument_parsing,306,0
2,failed,catalogue_validation,205,96
3,failed,reference_artifact_restore,306,96


Input contract,Observed value
Actual catalogue items validated,"1,852,162"
IDs sorted,False
Embedding row order,preserved from trained vocabulary
Catalogue check,passed
Reference count parts verified,96


Recovery check,Observed value
Root tests passed,229
Worker tests passed,126
Worker Python warning policy,errors in pytest configuration
Production S3 restore,passed
Recomputed saved reference part,False
AWS container loader warning,unresolved in managed image


## Evidence at a glance

Official Recall@20 evaluates the ordered recommendation list. Candidate ceilings describe the best a later ranker could recover from a larger pool. They answer different questions.

In [3]:
previous = json.loads((ROOT / "reports/metrics/two_tower_fold0_retrieval.json").read_text())
exact20 = next(row for row in previous["points"] if row["neural_k"] == 20)
union800 = next(row for row in previous["points"] if row["neural_k"] == 800)
ann_state = "Awaiting managed measurement" if report is None else (
    "Confirmation target met" if report["confirmation_fidelity_passed"]
    else "Confirmation target not met")
cards = [
    ("EXACT NEURAL · OFFICIAL RECALL@20", f"{exact20['weighted_neural_ceiling']:.3%}",
     "Measured across 103,468 held-out sessions"),
    ("ADDITIONAL CANDIDATE CEILING · K=800",
     f"+{100 * union800['weighted_incremental_ceiling']:.3f} pp",
     "Against the frozen baseline; ideal ranking ceiling"),
    ("ANN EXPERIMENT", ann_state, "Full-catalogue quality and CPU search latency"),
]
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:14px">' + ''.join(
    f'<div style="flex:1;min-width:240px;background:#f7f9fc;border:1px solid #dce3ec;'
    f'border-radius:12px;padding:20px"><div style="color:#536174;font-size:11px;'
    f'font-weight:700">{title}</div><div style="color:#17324f;font-size:26px;'
    f'font-weight:700;margin:12px 0">{value}</div><div style="color:#536174;'
    f'font-size:12px">{note}</div></div>' for title, value, note in cards) + '</div>'))


## Freeze the decision before looking at the result

The smallest `nprobe` meeting the **prospective 98% mean top-800 overlap target for each objective** is selected using tuning queries only. That one setting is evaluated on the disjoint confirmation queries. A failed target is an informative experiment; no extra folds or automatic retries are launched. Full-fold ANN export runs only after confirmation passes.

Fold 0 already selected the model checkpoint. The reserved ANN confirmation split does not make this an untouched model test. Full-fold metrics include the ANN tuning sessions.

In [4]:
display(pd.DataFrame([
    ("Tuning / confirmation sessions", f"{configuration['sample_sessions']//2:,} / {configuration['sample_sessions']//2:,}"),
    ("Index", f"IVFFlat · {configuration['nlist']:,} centroids · original FP32 vectors"),
    ("Training vectors / iterations", f"{configuration['train_items']:,} / {configuration['train_iterations']}"),
    ("Probe sweep", str(configuration["probes"])),
    ("Per-objective target", f"{configuration['target_overlap']:.0%} mean top-800 overlap"),
    ("Search CPU threads / batch size", f"{configuration['threads']} / {configuration['batch_size']}"),
    ("Latency sample", f"{configuration['latency_queries']} queries × {configuration['latency_repeats']} repeats; {configuration['warmup_queries']} warm-up calls"),
    ("Full-fold export", "Enabled after confirmation passes"),
], columns=["Prospective contract", "Setting"]).style.hide(axis="index"))


Prospective contract,Setting
Tuning / confirmation sessions,"2,048 / 2,048"
Index,"IVFFlat · 1,024 centroids · original FP32 vectors"
Training vectors / iterations,"65,536 / 20"
Probe sweep,"[32, 64, 128, 256]"
Per-objective target,98% mean top-800 overlap
Search CPU threads / batch size,4 / 128
Latency sample,128 queries × 3 repeats; 32 warm-up calls
Full-fold export,Enabled after confirmation passes


## Primary model quality: the official project metric

For each objective, sum top-20 positive hits over sessions and divide by the sum of `min(20, true-item count)`. Combine objectives using the [official OTTO weights](https://github.com/otto-de/recsys-dataset/blob/main/KAGGLE.md): **clicks 0.10, carts 0.30, orders 0.60**. Unknown catalogue positives remain misses.

NDCG@20, MRR@20, hit rate, and precision are diagnostics averaged over labeled sessions separately for each objective. They are not replacements for the official metric. The bootstrap resamples whole sessions jointly across objectives; its interval does not account for model selection.

In [5]:
if report is None:
    display(pd.DataFrame([
        {"Objective": o.title(), "Exact neural Recall@20": exact20["objectives"][o]["neural_ceiling"],
         "ANN Recall@20": "Pending"} for o in OBJECTIVES
    ]).style.format({"Exact neural Recall@20": "{:.3%}"}).hide(axis="index"))
    print("NDCG@20, MRR@20 and ANN differences will be read from the managed report.")
else:
    cohorts = [("Full-fold exact", report["full_reference_ranking"])]
    if report.get("full_ann_ranking"):
        cohorts.append(("Full-fold ANN", report["full_ann_ranking"]))
    if report.get("confirmation"):
        cohorts.extend([
            ("Confirmation exact", report["confirmation"]["exact_ranking"]),
            ("Confirmation ANN", report["confirmation"]["ranking"]),
        ])
    display(pd.DataFrame([
        {"Cohort / method": name, "Sessions": data["sessions"],
         "Official weighted Recall@20": data["weighted_recall_at_20"]}
        for name, data in cohorts
    ]).style.format({"Official weighted Recall@20": "{:.3%}", "Sessions": "{:,}"}).hide(axis="index"))
    diagnostics = []
    for name, data in cohorts:
        for objective, values in data["objectives"].items():
            diagnostics.append({"Cohort / method": name, "Objective": objective,
                **{key: values[key] for key in ("recall_at_20", "ndcg_at_20", "mrr_at_20",
                                               "hit_rate_at_20", "precision_at_20")}})
    display(pd.DataFrame(diagnostics).style.format({key: "{:.4f}" for key in diagnostics[0]
            if key not in {"Cohort / method", "Objective"}}).hide(axis="index"))
    if report.get("full_ann_paired_uncertainty"):
        lo, hi = report["full_ann_paired_uncertainty"]["weighted_recall_at_20_delta_ci95"]
        delta = report["full_ann_ranking"]["weighted_recall_at_20"] - report["full_reference_ranking"]["weighted_recall_at_20"]
        print(f"Full-fold ANN − exact: {delta*100:+.3f} pp; paired 95% interval [{lo*100:+.3f}, {hi*100:+.3f}] pp")


NDCG@20, MRR@20 and ANN differences will be read from the managed report.


Objective,Exact neural Recall@20,ANN Recall@20
Clicks,33.397%,Pending
Carts,21.225%,Pending
Orders,20.266%,Pending


## Search frontier: fidelity versus latency

Each point is one **tuning** configuration. Latency is warm, batch-1 CPU search plus FP32 reranking of precomputed queries. It excludes the session encoder, network, and index loading. The chart must not be described as end-to-end serving latency.

The exact CPU reference uses Flat inner-product search on the same tuning queries and the same thread count. The confirmation result appears separately below; it never chooses the configuration. FAISS explains the [IVF probe trade-off](https://github.com/facebookresearch/faiss/wiki/Faster-search).

In [6]:
if report is None:
    print("Search frontier pending. No latency or ANN overlap values have been measured for the full catalogue.")
else:
    frontier = []
    for probe, result in report["tuning"].items():
        for objective in OBJECTIVES:
            search = result["search"][objective]
            frontier.append({"nprobe": int(probe), "objective": objective,
                "overlap": search["fidelity"][str(report["contract"]["settings"]["candidate_depth"])],
                "p95_ms": search["latency"]["p95_ms"],
                "queries_per_second": search["batch_throughput_queries_per_second"]})
    frontier = pd.DataFrame(frontier)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
    for ax, objective in zip(axes, OBJECTIVES):
        rows = frontier[frontier.objective == objective].sort_values("nprobe")
        ax.plot(rows.p95_ms, rows.overlap, "o-", color=COLORS[objective], linewidth=2)
        for row in rows.itertuples():
            ax.annotate(str(row.nprobe), (row.p95_ms, row.overlap),
                        xytext=(5, 6), textcoords="offset points", fontsize=9)
        ax.axhline(report["contract"]["settings"]["target_overlap"], color="#758397",
                   linestyle="--", linewidth=1)
        ax.set_title(objective.title(), loc="left", fontweight="bold")
        ax.set_xlabel("Warm batch-1 p95 search (ms)")
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(PercentFormatter(1))
    axes[0].set_ylabel("Mean exact top-K ID overlap")
    fig.suptitle("OTTO | ANN tuning frontier", x=0.06, ha="left", fontweight="bold", fontsize=18, color="#17324f")
    fig.text(0.06, 0.015, "Point labels = nprobe · Dashed line = prospective fidelity target\nCPU search + reranking only; encoding and network are excluded.", fontsize=9, color="#536174")
    fig.subplots_adjust(left=0.07, right=0.98, top=0.80, bottom=0.24, wspace=0.22)
    figure_path = ROOT / "reports/figures/two_tower_ann.png"
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_path, dpi=180, facecolor=fig.get_facecolor())
    plt.show()
    display(frontier.style.format({"overlap": "{:.2%}", "p95_ms": "{:.3f}",
                                   "queries_per_second": "{:.1f}"}).hide(axis="index"))
    display(pd.DataFrame([
        {"Objective": o, "Exact CPU p50 ms": row["p50_ms"], "Exact CPU p95 ms": row["p95_ms"],
         "Exact CPU p99 ms": row["p99_ms"], "Timing observations": row["samples"]}
        for o, row in report["exact_cpu_latency_on_tuning_queries"].items()
    ]).style.format(precision=3).hide(axis="index"))


Search frontier pending. No latency or ANN overlap values have been measured for the full catalogue.


## Confirmation and engineering cost

A retained index or prediction part is reused only after its checksum and run identity match. Data is uploaded before its receipt. Missing/corrupt parts are recomputed; valid completed work survives across managed workers.

Latency observations can come from a previous compatible attempt. Retained compute time is the sum of committed part timings, not total paid time. AWS per-execution billable seconds live in the durable `control/executions/` records and are printed by the monitor.

In [7]:
if report is None:
    print("Confirmation, index sizes, memory use, and measured build/search times are pending.")
else:
    print(f"selected_nprobe={report['selected_nprobe']} confirmation_fidelity_passed={report['confirmation_fidelity_passed']}")
    if report.get("confirmation"):
        rows = []
        for objective, row in report["confirmation"]["search"].items():
            rows.append({"Objective": objective, "Overlap@20": row["fidelity"]["20"],
                "Overlap@K": row["fidelity"][str(report["contract"]["settings"]["candidate_depth"])],
                "p50 ms": row["latency"]["p50_ms"], "p95 ms": row["latency"]["p95_ms"],
                "p99 ms": row["latency"]["p99_ms"]})
        display(pd.DataFrame(rows).style.format({"Overlap@20": "{:.2%}", "Overlap@K": "{:.2%}",
            "p50 ms": "{:.3f}", "p95 ms": "{:.3f}", "p99 ms": "{:.3f}"}).hide(axis="index"))
    display(pd.DataFrame([
        {"Objective": o, "Index GiB": row["index_bytes"]/1024**3,
         "Retained build seconds": row["retained_build_compute_seconds"],
         "Load seconds this attempt": row["load_seconds_this_attempt"], "Index shards": row["shards"]}
        for o, row in report["index_builds"].items()
    ]).style.format(precision=3).hide(axis="index"))
    display(pd.DataFrame([
        ("Worker", report["contract"]["instance_type"]),
        ("Encoder device", report["contract"].get("encoder_device", "Unrecorded")),
        ("Search CPU threads", report["contract"]["settings"]["threads"]),
        ("Peak process RSS MiB", report["peak_rss_mib"]),
        ("Current attempt seconds", report["elapsed_seconds_this_attempt"]),
        ("Retained artifact compute seconds", report["retained_artifact_compute_seconds"]),
        ("Full-fold prediction export", report.get("prediction_export")),
    ], columns=["Resource / evidence", "Value"]).style.hide(axis="index"))


Confirmation, index sizes, memory use, and measured build/search times are pending.


## What would justify the next experiment?

1. Confirm that the selected configuration retains the prospective neighbor-overlap target and inspect its official Recall@20 change.
2. Assess CPU latency, throughput, memory, index size, and billable time at the disclosed workload.
3. Compare the exported full-fold ANN candidates against the frozen baseline. Neighbor overlap and exact-positive retention do **not** establish retained base-exclusive gain.
4. Publish the measured report and executed figures through a reviewed GitHub PR. Only then decide whether to generate the remaining OOF candidates and train budget-matched rankers.

The original exact-search gain remains measured evidence. ANN quality, latency, and complementary gain remain unmeasured until their corresponding workflows finish. None of these results alone establishes state-of-the-art performance.

In [8]:
runtime = {"python": platform.python_version(), "numpy": np.__version__,
           "pandas": pd.__version__}
print(json.dumps(runtime, sort_keys=True))
print(f"[{datetime.now(timezone.utc).isoformat()}] notebook_complete "
      f"elapsed_seconds={time.perf_counter()-STARTED:.3f}")


{"numpy": "2.5.3", "pandas": "3.0.5", "python": "3.12.13"}
[2026-09-06T22:15:29.959242+00:00] notebook_complete elapsed_seconds=0.102
